<a href="https://colab.research.google.com/github/salochaud/DatosMasivos/blob/main/Tarea_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dataset utilizado fifa 22 kaggle

In [1]:
!pip install pyspark -q

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, max, min, round, when, count

spark = SparkSession.builder \
    .appName("Tarea 1 - FIFA Players") \
    .getOrCreate()

In [9]:
df_players = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/players_22.csv")

In [10]:
df_players.show(5)
df_players.printSchema()

+---------+--------------------+-----------------+--------------------+----------------+-------+---------+---------+--------+---+----------+---------+---------+------------+-------------------+--------------------+------------+-------------+------------------+----------------+-----------+-------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+-----------+---------+---------+------------------+--------------------+--------------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+----------------+--------------------+----------

In [11]:
df_players.select(
    "overall", "potential", "age", "height_cm", "weight_kg",
    "pace", "shooting", "passing", "dribbling", "defending", "physic"
).describe().show()

+-------+-----------------+-----------------+------------------+------------------+-----------------+------------------+------------------+-----------------+-----------------+-----------------+-----------------+
|summary|          overall|        potential|               age|         height_cm|        weight_kg|              pace|          shooting|          passing|        dribbling|        defending|           physic|
+-------+-----------------+-----------------+------------------+------------------+-----------------+------------------+------------------+-----------------+-----------------+-----------------+-----------------+
|  count|            19239|            19239|             19239|             19239|            19239|             17107|             17107|            17107|            17107|            17107|            17107|
|   mean|65.77218150631529|71.07937002962731|25.210821768283175|181.29970372680492|74.94303238214044| 68.21307067282399|  52.3452972467411|57.3125621090

In [12]:
jugadores_elite = df_players.filter(col("overall") >= 85) \
    .select("short_name", "club_name", "nationality_name", "overall", "potential", "age")

jugadores_elite.orderBy(col("overall").desc()).show(10, truncate=False)

+-----------------+-------------------+----------------+-------+---------+---+
|short_name       |club_name          |nationality_name|overall|potential|age|
+-----------------+-------------------+----------------+-------+---------+---+
|L. Messi         |Paris Saint-Germain|Argentina       |93     |93       |34 |
|R. Lewandowski   |FC Bayern München  |Poland          |92     |92       |32 |
|Cristiano Ronaldo|Manchester United  |Portugal        |91     |91       |36 |
|Neymar Jr        |Paris Saint-Germain|Brazil          |91     |91       |29 |
|K. De Bruyne     |Manchester City    |Belgium         |91     |91       |30 |
|J. Oblak         |Atlético de Madrid |Slovenia        |91     |93       |28 |
|K. Mbappé        |Paris Saint-Germain|France          |91     |95       |22 |
|M. Neuer         |FC Bayern München  |Germany         |90     |90       |35 |
|M. ter Stegen    |FC Barcelona       |Germany         |90     |92       |29 |
|H. Kane          |Tottenham Hotspur  |England      

In [13]:
mexicanos = df_players.filter(col("nationality_name") == "Mexico") \
    .select("short_name", "club_name", "overall", "potential", "player_positions", "age")

mexicanos.orderBy(col("overall").desc()).show(15, truncate=False)

+-------------+-------------------------+-------+---------+----------------+---+
|short_name   |club_name                |overall|potential|player_positions|age|
+-------------+-------------------------+-------+---------+----------------+---+
|C. Vela      |Los Angeles FC           |83     |83       |RW, LW, CAM     |32 |
|R. Jiménez   |Wolverhampton Wanderers  |83     |83       |ST              |30 |
|J. Corona    |FC Porto                 |82     |82       |RM, RB, RW      |28 |
|H. Herrera   |Atlético de Madrid       |81     |81       |CM              |31 |
|H. Lozano    |Napoli                   |81     |82       |RW, LW          |25 |
|G. Ochoa     |Club América             |80     |80       |GK              |35 |
|A. Talavera  |Club Universidad Nacional|78     |78       |GK              |38 |
|A. Guardado  |Real Betis Balompié      |78     |78       |CDM, CM, LM     |34 |
|J. Hernández |LA Galaxy                |78     |78       |ST              |33 |
|L. Rodríguez |Tigres U.A.N.

In [14]:
mbappe = df_players.filter(col("short_name").contains("Mbapp"))
mbappe.select(
    "short_name", "club_name", "nationality_name", "overall", "potential",
    "pace", "shooting", "passing", "dribbling", "defending", "physic"
).show(truncate=False)

+----------+-------------------+----------------+-------+---------+----+--------+-------+---------+---------+------+
|short_name|club_name          |nationality_name|overall|potential|pace|shooting|passing|dribbling|defending|physic|
+----------+-------------------+----------------+-------+---------+----+--------+-------+---------+---------+------+
|K. Mbappé |Paris Saint-Germain|France          |91     |95       |97  |88      |80     |92       |36       |77    |
+----------+-------------------+----------------+-------+---------+----+--------+-------+---------+---------+------+



In [15]:
df_con_ipc = df_players.withColumn(
    "IPC",
    round(
        (col("potential") - col("overall")) / (100 - col("overall")) * 100,
        2
    )
)

promesas = df_con_ipc.filter((col("age") <= 23) & (col("overall") >= 70)) \
    .select("short_name", "age", "overall", "potential", "IPC", "club_name", "nationality_name")

promesas.orderBy(col("IPC").desc()).show(10, truncate=False)

+-------------------+---+-------+---------+-----+--------------------+----------------+
|short_name         |age|overall|potential|IPC  |club_name           |nationality_name|
+-------------------+---+-------+---------+-----+--------------------+----------------+
|Ansu Fati          |18 |76     |90       |58.33|FC Barcelona        |Spain           |
|N. Rovella         |19 |70     |87       |56.67|Genoa               |Italy           |
|R. Cherki          |17 |73     |88       |55.56|Olympique Lyonnais  |France          |
|M. Vandevoordt     |19 |71     |87       |55.17|KRC Genk            |Belgium         |
|R. Gravenberch     |19 |78     |90       |54.55|Ajax                |Netherlands     |
|G. Raspadori       |21 |74     |88       |53.85|U.S. Sassuolo Calcio|Italy           |
|F. Pellistri       |19 |70     |86       |53.33|Deportivo Alavés    |Uruguay         |
|Francisco Conceição|18 |70     |86       |53.33|FC Porto            |Portugal        |
|Pedri              |18 |81     

In [16]:
df_scores = df_players.withColumn(
    "score_ofensivo",
    round((col("pace") + col("shooting") + col("dribbling")) / 3, 2)
).withColumn(
    "score_defensivo",
    round((col("defending") + col("physic")) / 2, 2)
).withColumn(
    "perfil",
    when(col("score_ofensivo") > col("score_defensivo"), "Ofensivo")
    .otherwise("Defensivo")
)

df_scores.select(
    "short_name", "player_positions", "overall",
    "score_ofensivo", "score_defensivo", "perfil"
).orderBy(col("overall").desc()).show(15, truncate=False)

+-----------------+----------------+-------+--------------+---------------+---------+
|short_name       |player_positions|overall|score_ofensivo|score_defensivo|perfil   |
+-----------------+----------------+-------+--------------+---------------+---------+
|L. Messi         |RW, ST, CF      |93     |90.67         |49.5           |Ofensivo |
|R. Lewandowski   |ST              |92     |85.33         |63.0           |Ofensivo |
|Cristiano Ronaldo|ST, LW          |91     |89.67         |54.5           |Ofensivo |
|Neymar Jr        |LW, CAM         |91     |89.33         |50.0           |Ofensivo |
|K. De Bruyne     |CM, CAM         |91     |83.33         |71.0           |Ofensivo |
|J. Oblak         |GK              |91     |NULL          |NULL           |Defensivo|
|K. Mbappé        |ST, LW          |91     |92.33         |56.5           |Ofensivo |
|M. Neuer         |GK              |90     |NULL          |NULL           |Defensivo|
|M. ter Stegen    |GK              |90     |NULL      

In [17]:
por_pais = df_players.groupBy("nationality_name").agg(
    count("*").alias("num_jugadores"),
    round(avg("overall"), 2).alias("promedio_overall"),
    round(avg("potential"), 2).alias("promedio_potencial"),
    max("overall").alias("mejor_overall")
)

por_pais.orderBy(col("num_jugadores").desc()).show(10, truncate=False)

+----------------+-------------+----------------+------------------+-------------+
|nationality_name|num_jugadores|promedio_overall|promedio_potencial|mejor_overall|
+----------------+-------------+----------------+------------------+-------------+
|England         |1719         |63.93           |70.73             |90           |
|Germany         |1214         |65.63           |71.31             |90           |
|Spain           |1086         |69.56           |74.88             |88           |
|France          |980          |67.54           |73.42             |91           |
|Argentina       |960          |68.72           |73.11             |93           |
|Brazil          |897          |70.85           |72.97             |91           |
|Japan           |546          |64.41           |68.44             |79           |
|Netherlands     |439          |67.46           |73.96             |89           |
|United States   |413          |63.19           |70.18             |82           |
|Pol